# Q2 — Chuẩn hóa và Dropout

Khi mạng sâu hơn, việc tối ưu trở nên khó hơn:

- phân phối của các kích hoạt có thể thay đổi mạnh giữa các lớp;
- gradient có thể quá nhỏ hoặc quá lớn;
- mô hình có nhiều tham số dễ quá khớp.

Hai nhóm kỹ thuật thường được dùng để xử lý các vấn đề này:

1. **chuẩn hóa (normalization)** — ví dụ Batch Normalization;
2. **regularization** — ví dụ Dropout.

Trong notebook này, bạn sẽ tự cài đặt:

- Batch Normalization truyền xuôi;
- Batch Normalization truyền ngược;
- Layer Normalization truyền xuôi;
- Dropout truyền xuôi và truyền ngược.

Cuối notebook, bạn sẽ dùng PyTorch để quan sát tác động của Batch Normalization và
Dropout trong một thí nghiệm huấn luyện nhỏ.

In [ ]:
from pathlib import Path

assert Path("dl2").exists(), (
    "Không tìm thấy thư mục dl2/. "
    "Hãy chạy notebook từ thư mục gốc của Assignment 2."
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from dl2.gradcheck import (
    numerical_gradient_array,
    relative_error,
)
from dl2.data import get_fashion_mnist_loaders

np.random.seed(42)
torch.manual_seed(42)

## 1. Chuẩn hóa theo lô (Batch Normalization)

Với một mini-batch:

$$
X\in\mathbb{R}^{N\times D},
$$

Batch Normalization chuẩn hóa từng đặc trưng theo thống kê của mini-batch.

### Bước 1 — Trung bình

$$
\mu
=
\frac{1}{N}
\sum_{i=1}^{N}x_i.
$$

### Bước 2 — Phương sai

$$
\sigma^2
=
\frac{1}{N}
\sum_{i=1}^{N}(x_i-\mu)^2.
$$

### Bước 3 — Chuẩn hóa

$$
\hat{x}_i
=
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}.
$$

### Bước 4 — Co giãn và dịch chuyển

$$
y_i
=
\gamma\hat{x}_i+\beta.
$$

$\gamma$ và $\beta$ là các tham số học được.

Nhờ đó, lớp phía sau không bị buộc phải nhận đầu vào có một scale cố định do lớp trước
tạo ra.

## 2. Chế độ huấn luyện và chế độ đánh giá

Batch Normalization hoạt động khác nhau ở hai chế độ.

### Khi huấn luyện

Dùng mean và variance của mini-batch hiện tại.

### Khi đánh giá

Không nên phụ thuộc vào batch hiện tại. Ta dùng các thống kê trung bình tích lũy:

- `running_mean`;
- `running_var`.

Vì vậy một implementation BatchNorm phải duy trì trạng thái qua nhiều mini-batch.

### Thực hành 1 — `batchnorm_forward`

In [ ]:
def batchnorm_forward(
    x,
    gamma,
    beta,
    bn_param,
):
    """
    x: (N, D)
    gamma, beta: (D,)

    bn_param:
        mode: "train" hoặc "test"
        eps
        momentum
        running_mean
        running_var
    """
    mode = bn_param.get(
        "mode",
        "train",
    )
    eps = bn_param.get(
        "eps",
        1e-5,
    )
    momentum = bn_param.get(
        "momentum",
        0.9,
    )

    N, D = x.shape

    running_mean = bn_param.get(
        "running_mean",
        np.zeros(D),
    )
    running_var = bn_param.get(
        "running_var",
        np.zeros(D),
    )

    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Train:
    # - tính mean/variance của mini-batch;
    # - chuẩn hóa;
    # - scale/shift bằng gamma, beta;
    # - cập nhật running_mean/running_var.
    #
    # Test:
    # - dùng running statistics.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

    bn_param["running_mean"] = running_mean
    bn_param["running_var"] = running_var

    return out, cache

### Kiểm tra thống kê đầu ra

In [ ]:
x = (
    np.random.randn(200, 5)
    * np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    + np.array([0.0, 5.0, -3.0, 10.0, 2.0])
)

gamma = np.ones(5)
beta = np.zeros(5)

bn_param = {
    "mode": "train",
}

out, cache = batchnorm_forward(
    x,
    gamma,
    beta,
    bn_param,
)

print("Mean đầu ra:", out.mean(axis=0))
print("Variance đầu ra:", out.var(axis=0))

assert np.allclose(
    out.mean(axis=0),
    0.0,
    atol=1e-7,
)
assert np.allclose(
    out.var(axis=0),
    1.0,
    atol=1e-4,
)

print("✓ BatchNorm train-mode statistics hợp lý")

### Câu hỏi 1 — Kích thước lô (batch size)

Nếu batch size chỉ bằng 2 hoặc 4, mean và variance của từng batch có thể dao động mạnh.

Điều này có thể ảnh hưởng Batch Normalization như thế nào? Vì sao BatchNorm đôi khi kém
ổn định khi batch size quá nhỏ?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 3. Truyền ngược qua chuẩn hóa theo lô

BatchNorm không chỉ có phép chuẩn hóa. Mean và variance cũng phụ thuộc vào toàn bộ
mini-batch, vì vậy đạo hàm của một mẫu liên quan đến các mẫu khác trong batch.

Ta cần tính:

$$
dx,\qquad d\gamma,\qquad d\beta.
$$

Hai đạo hàm dễ thấy:

$$
d\beta
=
\sum_i dout_i,
$$

$$
d\gamma
=
\sum_i dout_i\hat{x}_i.
$$

Phần $dx$ phức tạp hơn vì phải truyền qua:

$$
x\rightarrow\mu,\sigma^2\rightarrow\hat{x}\rightarrow y.
$$

### Thực hành 2 — `batchnorm_backward`

In [ ]:
def batchnorm_backward(
    dout,
    cache,
):
    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Tính dx, dgamma, dbeta.
    # Có thể dùng quy tắc dây chuyền theo computation graph
    # hoặc dùng công thức rút gọn nếu bạn tự suy ra được.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

### Kiểm tra đạo hàm bằng vi phân số

In [ ]:
np.random.seed(1)

N, D = 4, 5

x = np.random.randn(N, D)
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {
    "mode": "train",
}

out, cache = batchnorm_forward(
    x,
    gamma,
    beta,
    bn_param,
)

dx, dgamma, dbeta = batchnorm_backward(
    dout,
    cache,
)

fx = lambda xx: batchnorm_forward(
    xx,
    gamma,
    beta,
    {"mode": "train"},
)[0]

fg = lambda gg: batchnorm_forward(
    x,
    gg,
    beta,
    {"mode": "train"},
)[0]

fb = lambda bb: batchnorm_forward(
    x,
    gamma,
    bb,
    {"mode": "train"},
)[0]

dx_num = numerical_gradient_array(
    fx,
    x.copy(),
    df=dout,
)

dgamma_num = numerical_gradient_array(
    fg,
    gamma.copy(),
    df=dout,
)

dbeta_num = numerical_gradient_array(
    fb,
    beta.copy(),
    df=dout,
)

print(
    "dx error:",
    relative_error(dx, dx_num),
)
print(
    "dgamma error:",
    relative_error(dgamma, dgamma_num),
)
print(
    "dbeta error:",
    relative_error(dbeta, dbeta_num),
)

assert relative_error(dx, dx_num) < 1e-6
assert relative_error(dgamma, dgamma_num) < 1e-8
assert relative_error(dbeta, dbeta_num) < 1e-8

print("✓ batchnorm_backward vượt qua kiểm tra")

## 4. Chuẩn hóa theo lớp (Layer Normalization)

Layer Normalization cũng chuẩn hóa dữ liệu nhưng khác BatchNorm ở trục thống kê.

Với BatchNorm, ta chuẩn hóa từng feature qua nhiều mẫu.

Với LayerNorm, ta chuẩn hóa **các feature trong cùng một mẫu**.

Vì vậy LayerNorm:

- không cần running mean/variance;
- không phụ thuộc batch size;
- dùng cùng công thức ở train và test.

Đây là lý do LayerNorm phù hợp với các mô hình chuỗi và Transformer, nơi batch size hoặc
sequence length có thể thay đổi.

### Thực hành 3 — `layernorm_forward`

In [ ]:
def layernorm_forward(
    x,
    gamma,
    beta,
    eps=1e-5,
):
    """
    x: (N, D)

    Chuẩn hóa theo chiều D cho từng mẫu.
    """
    # ============================================================
    # TODO: CODE CỦA BẠN
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
x = (
    np.random.randn(6, 8)
    * 4.0
    + 7.0
)

gamma = np.ones(8)
beta = np.zeros(8)

ln_out = layernorm_forward(
    x,
    gamma,
    beta,
)

print(
    "Mean theo từng mẫu:",
    ln_out.mean(axis=1),
)
print(
    "Variance theo từng mẫu:",
    ln_out.var(axis=1),
)

assert np.allclose(
    ln_out.mean(axis=1),
    0.0,
    atol=1e-7,
)
assert np.allclose(
    ln_out.var(axis=1),
    1.0,
    atol=1e-4,
)

print("✓ LayerNorm statistics hợp lý")

### Câu hỏi 2 — Chuẩn hóa theo lô và chuẩn hóa theo lớp

Nêu một tình huống bạn ưu tiên Layer Normalization thay vì Batch Normalization.

Giải thích dựa trên **trục được chuẩn hóa** và sự phụ thuộc vào batch size.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 5. Dropout

Dropout là một kỹ thuật regularization.

Trong lúc huấn luyện, mỗi activation được giữ lại với xác suất $p$.

Với **inverted dropout**, ta dùng mask:

$$
m_i
=
\frac{\text{Bernoulli}(p)}{p}.
$$

Đầu ra:

$$
y_i=x_i m_i.
$$

Nhờ chia cho $p$, kỳ vọng của đầu ra không đổi:

$$
E[y_i]=x_i.
$$

Khi đánh giá, không cần dropout nữa:

$$
y=x.
$$

### Thực hành 4 — `dropout_forward`

In [ ]:
def dropout_forward(
    x,
    dropout_param,
):
    """
    dropout_param:
        p: xác suất giữ activation
        mode: "train" hoặc "test"
        seed: tùy chọn
    """
    p = dropout_param.get(
        "p",
        0.5,
    )
    mode = dropout_param.get(
        "mode",
        "train",
    )

    seed = dropout_param.get(
        "seed",
        None,
    )

    if seed is not None:
        np.random.seed(seed)

    # ============================================================
    # TODO: CODE CỦA BẠN
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

    return out, cache

In [ ]:
x = np.ones(
    (10000, 10),
)

dropout_param = {
    "mode": "train",
    "p": 0.6,
    "seed": 42,
}

drop_out, drop_cache = dropout_forward(
    x,
    dropout_param,
)

fraction_nonzero = np.mean(
    drop_out != 0
)

print(
    "Tỷ lệ phần tử khác 0:",
    fraction_nonzero,
)
print(
    "Mean đầu ra:",
    drop_out.mean(),
)

assert abs(
    fraction_nonzero - 0.6
) < 0.02

assert abs(
    drop_out.mean() - 1.0
) < 0.03

print("✓ inverted dropout có kỳ vọng đúng")

### Thực hành 5 — `dropout_backward`

In [ ]:
def dropout_backward(
    dout,
    cache,
):
    # ============================================================
    # TODO: CODE CỦA BẠN
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

In [ ]:
dout = np.random.randn(
    *drop_out.shape
)

dx = dropout_backward(
    dout,
    drop_cache,
)

assert dx.shape == x.shape

# Vì x toàn số 1, vị trí đầu ra bằng 0 chính là vị trí bị dropout.
assert np.all(
    dx[drop_out == 0] == 0
)

print("✓ dropout_backward có kích thước và mask hợp lý")

### Câu hỏi 3 — Vì sao Dropout chỉ dùng khi huấn luyện?

Điều gì sẽ xảy ra nếu ta vẫn ngẫu nhiên loại neuron trong lúc đánh giá?

Giải thích vì sao inverted dropout giúp chế độ đánh giá trở nên đơn giản.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 6. Thí nghiệm nhỏ bằng PyTorch

Phần trên giúp hiểu cơ chế. Bây giờ ta dùng implementation chuẩn của PyTorch để quan sát
tác động trong huấn luyện thực tế.

Ta dùng Fashion-MNIST vì mô hình nhỏ có thể chạy nhanh trên cả CPU và GPU.

Ba cấu hình:

1. MLP cơ sở;
2. MLP + BatchNorm;
3. MLP + Dropout.

Mục tiêu **không phải** tìm cấu hình tốt nhất. Hãy quan sát:

- tốc độ giảm hàm mất mát (loss);
- độ chính xác huấn luyện;
- độ chính xác kiểm định;
- chênh lệch giữa train và validation.

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

train_loader, val_loader, _ = (
    get_fashion_mnist_loaders(
        batch_size=256,
        seed=42,
    )
)

print("Device:", device)

In [ ]:
class TinyMLP(nn.Module):
    def __init__(
        self,
        use_batchnorm=False,
        dropout_p=0.0,
    ):
        super().__init__()

        layers = [
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
        ]

        if use_batchnorm:
            layers.append(
                nn.BatchNorm1d(256)
            )

        layers.append(
            nn.ReLU()
        )

        if dropout_p > 0:
            layers.append(
                nn.Dropout(dropout_p)
            )

        layers.extend([
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        ])

        self.net = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.net(x)


def train_short_experiment(
    use_batchnorm=False,
    dropout_p=0.0,
    epochs=3,
):
    torch.manual_seed(42)

    model = TinyMLP(
        use_batchnorm=use_batchnorm,
        dropout_p=dropout_p,
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=0.08,
        momentum=0.9,
    )

    history = []

    for epoch in range(epochs):
        model.train()

        total = 0
        correct = 0
        loss_sum = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(
                logits,
                y,
            )
            loss.backward()
            optimizer.step()

            loss_sum += (
                loss.item()
                * len(x)
            )
            total += len(x)
            correct += (
                logits.argmax(dim=1) == y
            ).sum().item()

        model.eval()

        val_total = 0
        val_correct = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)

                val_total += len(x)
                val_correct += (
                    logits.argmax(dim=1) == y
                ).sum().item()

        history.append({
            "loss":
                loss_sum / total,
            "train_acc":
                correct / total,
            "val_acc":
                val_correct / val_total,
        })

    return history

### Thực hành 6 — So sánh ba cấu hình

In [ ]:
# ============================================================
# TODO
#
# Chạy ba thí nghiệm:
# 1. baseline
# 2. BatchNorm
# 3. Dropout với p hợp lý
#
# Tạo một bảng hoặc biểu đồ so sánh:
# - loss
# - train accuracy
# - validation accuracy
# ============================================================

raise NotImplementedError("Hoàn thiện thí nghiệm so sánh ba cấu hình")

### Câu hỏi 4 — Đọc kết quả thí nghiệm

Dựa trên kết quả của **chính bạn**:

1. BatchNorm có làm quá trình tối ưu nhanh hơn trong 3 epoch đầu không?
2. Dropout ảnh hưởng độ chính xác huấn luyện và độ chính xác kiểm định như thế nào?
3. Có nên kết luận một kỹ thuật luôn tốt hơn chỉ từ một thí nghiệm ngắn này không? Vì sao?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## Tóm tắt Q2

Bạn đã thực hành ba ý tưởng:

- Batch Normalization chuẩn hóa theo mini-batch và cần running statistics;
- Layer Normalization chuẩn hóa trong từng mẫu và không phụ thuộc batch size;
- Dropout regularize bằng cách ngẫu nhiên loại một phần activation khi huấn luyện.

Ở Q3, bạn sẽ dùng các module chuẩn hóa và Dropout của PyTorch trong một CNN thực sự.